# Лабораторна робота №5. Моніторинг моделі машинного навчання (AstroVision)

**Мета:** порівняти еталонні (reference) та виробничі (current) дані моделі класифікації зір, виявити data drift, target/prediction drift і деградацію якості за допомогою Evidently AI.

Перед запуском: сервер піднято (`uvicorn app.main:app --reload`), виконано `POST /train-model`, надіслано трафік:
`python src/simulate_traffic.py normal` та `python src/simulate_traffic.py drift`. Ноутбук запускати з папки `src/`, комірки — зверху вниз.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))          # корінь проєкту
import pandas as pd
from app.services import monitoring as m
pd.set_option("display.width", 200)

## 1. Підготовка даних
- **reference** — записи, на яких навчалась модель: `predictions` (`source='train'`) ⨝ `stars_data_cleaned` за `Star_ID` (9 000 рядків).
- **current** — запити до `/predict`: `inference_inputs` ⨝ `predictions` (`source='inference'`) за `input_id`.
- `target` = фактичний тип зорі (`true_label`), `prediction` = відповідь моделі. Обидва набори мають однакові стовпці: `timestamp`, 15 ознак, `target`, `prediction`.
- `Galaxy_Type` виключено з моніторингу: у даних єдине значення (Spiral), для константи дрейф не визначений.

In [18]:
print(m.load_reference.__doc__); print(m.load_current.__doc__)
ref = m.load_reference()
cur_normal = m.load_current("normal")
cur_drift = m.load_current("drift")
for name, d in [("reference", ref), ("current normal", cur_normal), ("current drift", cur_drift)]:
    print(f"{name}: {d.shape}, ground truth: {d['target'].notna().sum()}, "
          f"timestamp: {d['timestamp'].min()} — {d['timestamp'].max()}")
ref.head()

Reference = записи, на яких навчалась модель (predictions.source='train') разом з ознаками зі stars_data_cleaned.
Current = запити до /predict (inference_inputs + predictions.source='inference').
    target = true_label (відомий лише для запитів за star_id), для решти — NA.
    run: 'all' або назва прогону з logs/traffic_runs.json ('normal', 'drift') — відбір за часом запиту.
reference: (9000, 18), ground truth: 9000, timestamp: 2025-01-01 00:43:57 — 2025-12-30 23:14:40
current normal: (1000, 18), ground truth: 1000, timestamp: 2026-11-04 01:02:19 — 2026-11-04 01:02:41
current drift: (500, 18), ground truth: 500, timestamp: 2026-11-04 01:18:35 — 2026-11-04 01:18:46


,timestamp,Temperature_K,Luminosity_L_Sun,Radius_R_Sun,Mass_M_Sun,Distance_LY,Apparent_Magnitude,Absolute_Magnitude,Age_Million_Years,Metallicity,Rotation_Speed_KM_S,Surface_Gravity,Spectral_Class,Star_Color,Constellation,Variable_Star,target,prediction
0,2025-08-12 17:39:38,2.158972,1.893069,2.284568,2.301533,-0.698544,-2.299027,-1.785185,-1.417665,0.089973,-0.604275,-0.580795,1,1,9,0,5,5
1,2025-07-21 13:33:44,-0.670725,-0.601988,-0.804822,0.247138,-0.659598,-0.160986,0.147209,0.579620,-0.118767,-0.938784,2.082678,4,2,15,0,3,3
2,2025-06-19 20:26:02,-0.485520,1.893069,-0.162976,0.020517,-0.608626,-0.584014,-0.850845,-0.594676,-0.937667,2.296546,-0.204080,4,2,6,0,1,1
3,2025-07-19 03:43:50,0.040068,-0.562423,0.366408,-0.653165,1.998360,1.066623,0.001941,1.355538,1.575234,2.296546,-0.829760,2,6,6,0,3,3
4,2025-05-15 14:35:04,0.596856,-0.640791,-0.691738,-0.684412,-0.269692,0.926928,0.818264,0.206914,-0.142852,-0.832686,-0.179320,0,4,12,1,6,6


In [19]:
# структура однакова: стовпці, типи, пропуски
assert list(ref.columns) == list(cur_normal.columns) == list(cur_drift.columns)
pd.DataFrame({"reference": ref.dtypes.astype(str), "current": cur_drift.dtypes.astype(str),
              "NaN у current": cur_drift.isna().sum()})

,reference,current,NaN у current
timestamp,datetime64[ns],datetime64[ns],0
Temperature_K,float64,float64,0
Luminosity_L_Sun,float64,float64,0
Radius_R_Sun,float64,float64,0
Mass_M_Sun,float64,float64,0
Distance_LY,float64,float64,0
Apparent_Magnitude,float64,float64,0
Absolute_Magnitude,float64,float64,0
Age_Million_Years,float64,float64,0
Metallicity,float64,float64,0


> **Примітка щодо масштабу.** Ознаки зберігаються у вигляді після ЛР2: числові — z-оцінки (StandardScaler), категоріальні — коди LabelEncoder. Тому на графіках Evidently значення температури чи світності — це відхилення від середнього в одиницях σ, а не Кельвіни чи світності Сонця. Зсув у +1.5 для `Temperature_K` означає «на півтори σ гарячіші, ніж у навчальній вибірці».

## 2. Швидка перевірка дрейфу (scipy) — перехресна перевірка до звітів Evidently

In [20]:
print("=== normal ==="); display(m.drift_summary(ref, cur_normal).round(4))
print("=== drift ===");  display(m.drift_summary(ref, cur_drift).round(4))

=== normal ===


,feature,type,test,statistic,p_value,ref_mean,cur_mean,drift
0,Constellation,cat,chi2,17.3473,0.2985,NaN,NaN,False
1,Radius_R_Sun,num,KS,0.0308,0.3556,0.0019,-0.0169,False
2,Metallicity,num,KS,0.0298,0.3959,0.0008,-0.0072,False
3,Surface_Gravity,num,KS,0.0294,0.4099,-0.0035,0.0318,False
4,Variable_Star,cat,chi2,0.4832,0.4870,NaN,NaN,False
5,Age_Million_Years,num,KS,0.0276,0.4946,0.0029,-0.0265,False
6,Absolute_Magnitude,num,KS,0.0248,0.6310,0.0012,-0.0109,False
7,Luminosity_L_Sun,num,KS,0.0232,0.7095,-0.0014,0.0125,False
8,Distance_LY,num,KS,0.0210,0.8158,0.0028,-0.0254,False
9,Spectral_Class,cat,chi2,2.4994,0.8685,NaN,NaN,False


=== drift ===


,feature,type,test,statistic,p_value,ref_mean,cur_mean,drift
0,Luminosity_L_Sun,num,KS,0.7489,0.0000,-0.0014,1.0016,True
1,Temperature_K,num,KS,0.7388,0.0000,0.0005,1.4325,True
2,Absolute_Magnitude,num,KS,0.4992,0.0000,0.0012,-0.8101,True
3,Variable_Star,cat,chi2,484.8959,0.0000,NaN,NaN,True
4,Spectral_Class,cat,chi2,511.5379,0.0000,NaN,NaN,True
5,Metallicity,num,KS,0.3900,0.0000,0.0008,-1.0188,True
6,Surface_Gravity,num,KS,0.0596,0.0670,-0.0035,0.0894,False
7,Star_Color,cat,chi2,8.5897,0.1980,NaN,NaN,False
8,Radius_R_Sun,num,KS,0.0480,0.2188,0.0019,-0.0552,False
9,Age_Million_Years,num,KS,0.0389,0.4606,0.0029,-0.0424,False


In [21]:
print("Частки класів (normal)"); display(m.class_shares(ref, cur_normal))
print("Частки класів (drift)");  display(m.class_shares(ref, cur_drift))

Частки класів (normal)


,reference_target,current_target,current_prediction
0,0.0713,0.071,0.071
1,0.1192,0.119,0.119
2,0.0304,0.031,0.032
3,0.3534,0.354,0.354
4,0.2519,0.252,0.252
5,0.0762,0.076,0.075
6,0.0974,0.097,0.097


Частки класів (drift)


,reference_target,current_target,current_prediction
0,0.0713,0.07,0.00
1,0.1192,0.118,0.00
2,0.0304,0.03,0.05
3,0.3534,0.362,0.90
4,0.2519,0.268,0.00
5,0.0762,0.07,0.05
6,0.0974,0.082,0.00


In [22]:
print("normal"); display(m.performance_summary(ref, cur_normal))
print("drift");  display(m.performance_summary(ref, cur_drift))

normal


,набір,n,accuracy,f1_macro
0,reference (train),9000,1.000,1.0000
1,current (з ground truth),1000,0.991,0.9711


drift


,набір,n,accuracy,f1_macro
0,reference (train),9000,1.000,1.0000
1,current (з ground truth),500,0.434,0.2843


## 3. Дашборди Evidently
Для кожного прогону будуємо 4 звіти: Data Drift, Target Drift, Classification Performance, Data Quality. HTML зберігаються в `reports/`.

In [23]:
for run, cur in [("normal", cur_normal), ("drift", cur_drift)]:
    for kind in m.REPORT_TYPES:
        try:
            print("OK", m.build_report(kind, ref, cur, m.REPORTS_DIR / f"{kind}_{run}.html"))
        except Exception as e:
            print("ПОМИЛКА", run, kind, type(e).__name__, e)

OK /Users/sofiadumich/Desktop/lab5_project/reports/drift_normal.html
OK /Users/sofiadumich/Desktop/lab5_project/reports/target_normal.html
OK /Users/sofiadumich/Desktop/lab5_project/reports/performance_normal.html
OK /Users/sofiadumich/Desktop/lab5_project/reports/quality_normal.html
OK /Users/sofiadumich/Desktop/lab5_project/reports/drift_drift.html
OK /Users/sofiadumich/Desktop/lab5_project/reports/target_drift.html
OK /Users/sofiadumich/Desktop/lab5_project/reports/performance_drift.html


/Users/sofiadumich/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/Users/sofiadumich/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/Users/sofiadumich/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/Users/sofiadumich/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no 

OK /Users/sofiadumich/Desktop/lab5_project/reports/quality_drift.html


In [24]:
from IPython.display import IFrame
IFrame(str(m.REPORTS_DIR / "drift_drift.html"), width="100%", height=700)

## Висновок


- **Data drift.** У прогоні `normal` (1 000 запитів) дрейфу немає: за KS/χ² для всіх 15 ознак p > 0,29. У прогоні `drift` (500 запитів) дрейф виявлено в 6 із 15 ознак: `Luminosity_L_Sun`, `Temperature_K`, `Absolute_Magnitude`, `Metallicity`, `Spectral_Class`, `Variable_Star` (p < 0,001); решта ознак стабільні.
- **Target / prediction drift.** Частки фактичних класів майже не змінились (відхилення до 2 п. п.), а частка передбаченого класу Main Sequence зросла з ≈ 35 % до 89 %; класи Red Dwarf, Brown Dwarf, White Dwarf майже зникли з відповідей. Це prediction drift при стабільному target.
- **Якість моделі.** accuracy на reference (train) 0,997, на `normal` 0,985, на `drift` 0,444 (macro-F1 0,306). Це явна деградація.
- **Чи потрібне оновлення.** Для `normal` — ні, модель стабільна. Для `drift` — так: перенавчання на свіжих даних і налаштування алертів (наприклад, drift share > 30 % або accuracy < 0,9).
- **Обмеження.** Дрейф у прогоні `drift` змодельований (зсув ознак у σ), а не спостережений у виробництві; reference-accuracy обчислена на train, тому завищена порівняно з hold-out (0,985).
- **Проблеми під час виконання.** Опишіть свої.